# Notebook 11 v4 — Final manuscript tables and figures

## Grain AI–MDPI

Este notebook consolida los resultados aprobados de los notebooks anteriores y genera los materiales finales para el artículo:

- Tablas principales en CSV, Markdown y LaTeX.
- Figuras principales en PNG y PDF.
- Tablas suplementarias.
- Registro trazable de afirmaciones numéricas.
- Índice de procedencia de archivos.
- Validaciones de coherencia.
- Paquete ZIP final.

### Principio metodológico

Este notebook **no entrena modelos, no recalibra probabilidades, no vuelve a seleccionar campeones y no calcula nuevos umbrales**. Utiliza exclusivamente resultados ya congelados y validados.

### Estructura editorial propuesta

1. Características de los conjuntos y particiones.
2. Rendimiento de los campeones.
3. Ablación tamaño–forma.
4. Clasificación selectiva.
5. Predicción conformal.
6. Interpretabilidad morfométrica.
7. Sensibilidad ante incertidumbre de medición.

## Editorial v4

- All main figure labels are in English.
- Measurement-uncertainty results are separated into Table 7A and Table 7B.
- The claim register includes interpretation and simulated measurement-sensitivity claims.
- Internal representation and block codes are replaced by publication-ready labels in figures.


In [ ]:
# ============================================================
# 1. Dependencias y entorno
# ============================================================
import importlib
import subprocess
import sys

for module_name, package_name in {
    "pyarrow": "pyarrow",
    "tabulate": "tabulate",
}.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import re
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("CPU disponibles:", os.cpu_count())

In [ ]:
# ============================================================
# 2. Configuración
# ============================================================
RANDOM_SEED = 20260808

DATASETS = ["dry_bean", "iniap_rice"]

DATASET_DISPLAY_NAMES = {
    "dry_bean": "Dry Bean",
    "iniap_rice": "INIAP Rice",
}

REPRESENTATION_ORDER = [
    "R1_full_native",
    "R2_measured_core",
    "R3_invariant_core",
    "R4_hybrid_log_area",
]

LOCAL_ROOT = Path("/content/Grain_AI_MDPI_notebook11")
INPUT_DIR = LOCAL_ROOT / "00_input"
EXTRACT_ROOT = LOCAL_ROOT / "01_extracted"
OUTPUT_ROOT = LOCAL_ROOT / "02_outputs"

PROTOCOL_DIR = OUTPUT_ROOT / "03_protocol"
RESULTS_DIR = OUTPUT_ROOT / "07_results" / "11_MANUSCRIPT_ASSETS"
FIGURES_DIR = OUTPUT_ROOT / "08_figures" / "11_MANUSCRIPT_ASSETS"
TABLES_DIR = OUTPUT_ROOT / "09_tables" / "11_MANUSCRIPT_ASSETS"
SUPPLEMENT_DIR = OUTPUT_ROOT / "10_supplementary"
TEXT_DIR = OUTPUT_ROOT / "11_manuscript_text"

for directory in [
    INPUT_DIR,
    EXTRACT_ROOT,
    PROTOCOL_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    SUPPLEMENT_DIR,
    TEXT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

ZIP_FILENAMES = {
    "notebook02": "NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip",
    "notebook03": "NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip",
    "notebook04": "NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip",
    "notebook05": "NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip",
    "notebook06": "NOTEBOOK06_PAIRED_INFERENCE_OUTPUTS.zip",
    "notebook07": "NOTEBOOK07_SELECTIVE_UNCERTAINTY_OUTPUTS.zip",
    "notebook08_v2": "NOTEBOOK08_V2_CANONICAL_CONFORMAL_OUTPUTS.zip",
    "notebook09": "NOTEBOOK09_ABLATION_INTERPRETABILITY_OUTPUTS.zip",
    "notebook10": "NOTEBOOK10_MEASUREMENT_UNCERTAINTY_OUTPUTS.zip",
}

print("ZIP requeridos:", len(ZIP_FILENAMES))

In [ ]:
# ============================================================
# 3. Google Drive, localización y extracción
# ============================================================
try:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    files = None
    print("No se detectó Google Colab.")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Grain_Robustness_Q1")
DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

def find_exact_file(root: Path, filename: str):
    if not root.exists():
        return None

    matches = sorted(root.rglob(filename))

    if not matches:
        return None

    return max(
        matches,
        key=lambda path: path.stat().st_mtime,
    )

def obtain_input_zip(filename: str) -> Path:
    found = find_exact_file(
        DRIVE_PROJECT_ROOT,
        filename,
    )

    if found is not None:
        print("Encontrado:", found)
        return found

    local_candidate = Path("/content") / filename

    if local_candidate.exists():
        print("Encontrado:", local_candidate)
        return local_candidate

    if IN_COLAB and files is not None:
        print(
            f"No se encontró {filename}. "
            "Selecciónelo para subirlo."
        )

        uploaded = files.upload()

        if filename not in uploaded:
            raise FileNotFoundError(
                f"Debe subir exactamente {filename}."
            )

        destination = INPUT_DIR / filename
        destination.write_bytes(uploaded[filename])
        return destination

    raise FileNotFoundError(
        f"No se encontró {filename}."
    )

def clean_extract(
    zip_path: Path,
    destination: Path,
):
    if destination.exists():
        shutil.rmtree(destination)

    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as archive:
        archive.extractall(destination)

zip_paths = {}
extracted_roots = {}

for key, filename in ZIP_FILENAMES.items():
    zip_path = obtain_input_zip(filename)
    destination = EXTRACT_ROOT / key

    clean_extract(
        zip_path,
        destination,
    )

    zip_paths[key] = zip_path
    extracted_roots[key] = destination

display(
    pd.DataFrame(
        [
            {
                "source": key,
                "zip_path": str(zip_paths[key]),
                "extract_path": str(extracted_roots[key]),
            }
            for key in ZIP_FILENAMES
        ]
    )
)

In [ ]:
# ============================================================
# 4. Funciones de búsqueda, carga y exportación
# ============================================================
def normalized_name(path: Path):
    return path.name.lower().replace("-", "_")

def find_csv(
    root: Path,
    exact_name=None,
    tokens=None,
    required=True,
):
    candidates = list(root.rglob("*.csv"))

    if exact_name is not None:
        exact_matches = [
            path
            for path in candidates
            if path.name == exact_name
        ]

        if exact_matches:
            return sorted(
                exact_matches,
                key=lambda path: (
                    len(path.parts),
                    str(path),
                ),
            )[0]

    if tokens:
        normalized_tokens = [
            token.lower().replace("-", "_")
            for token in tokens
        ]

        token_matches = [
            path
            for path in candidates
            if all(
                token in normalized_name(path)
                for token in normalized_tokens
            )
        ]

        if token_matches:
            return sorted(
                token_matches,
                key=lambda path: (
                    len(path.name),
                    len(path.parts),
                    str(path),
                ),
            )[0]

    if required:
        raise FileNotFoundError(
            f"No se encontró CSV en {root}. "
            f"exact_name={exact_name}, tokens={tokens}"
        )

    return None

def find_parquets(
    root: Path,
    name_suffix="_with_split.parquet",
):
    return sorted(
        [
            path
            for path in root.rglob("*.parquet")
            if path.name.endswith(
                name_suffix
            )
        ]
    )

def read_csv_source(
    source_key,
    exact_name=None,
    tokens=None,
    required=True,
):
    path = find_csv(
        extracted_roots[source_key],
        exact_name=exact_name,
        tokens=tokens,
        required=required,
    )

    if path is None:
        return None, None

    return pd.read_csv(path), path

def first_existing_column(
    frame,
    candidates,
    required=True,
):
    for column in candidates:
        if column in frame.columns:
            return column

    if required:
        raise KeyError(
            f"No se encontró ninguna columna entre: {candidates}. "
            f"Disponibles: {list(frame.columns)}"
        )

    return None

def format_value(value, digits=4):
    if pd.isna(value):
        return ""

    if isinstance(value, (bool, np.bool_)):
        return "Yes" if value else "No"

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)):
        return f"{float(value):.{digits}f}"

    return str(value)

def export_table(
    frame,
    base_name,
    destination=TABLES_DIR,
    digits=4,
):
    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    csv_path = destination / f"{base_name}.csv"
    markdown_path = destination / f"{base_name}.md"
    latex_path = destination / f"{base_name}.tex"

    frame.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig",
    )

    formatted = frame.copy()

    for column in formatted.columns:
        formatted[column] = formatted[
            column
        ].map(
            lambda value: format_value(
                value,
                digits=digits,
            )
        )

    markdown_path.write_text(
        formatted.to_markdown(index=False),
        encoding="utf-8",
    )

    latex_path.write_text(
        formatted.to_latex(
            index=False,
            escape=True,
        ),
        encoding="utf-8",
    )

    return {
        "csv": csv_path,
        "markdown": markdown_path,
        "latex": latex_path,
    }

def copy_source_table(
    source_path: Path,
    target_name: str,
):
    destination = (
        SUPPLEMENT_DIR
        / f"{target_name}{source_path.suffix}"
    )

    shutil.copy2(
        source_path,
        destination,
    )

    return destination

def save_figure(
    figure,
    base_name,
):
    png_path = FIGURES_DIR / f"{base_name}.png"
    pdf_path = FIGURES_DIR / f"{base_name}.pdf"

    figure.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    return png_path, pdf_path

In [ ]:
# ============================================================
# 5. Carga de archivos fuente y validaciones
# ============================================================
selected_configurations, selected_config_path = read_csv_source(
    "notebook04",
    exact_name="03_selected_model_configurations.csv",
    tokens=["selected", "model", "configuration"],
)

champion_intervals, champion_intervals_path = read_csv_source(
    "notebook06",
    exact_name="Table_training_selected_champions_with_test_ci.csv",
    tokens=["training", "selected", "champion", "test", "ci"],
)

representation_contrasts, representation_contrasts_path = read_csv_source(
    "notebook06",
    exact_name="Table_paired_representation_contrasts.csv",
    tokens=["paired", "representation", "contrast"],
)

model_metric_intervals, model_metric_intervals_path = read_csv_source(
    "notebook06",
    exact_name="Table_model_metric_intervals.csv",
    tokens=["model", "metric", "interval"],
)

selective_operating_points, selective_operating_points_path = read_csv_source(
    "notebook07",
    exact_name="Table_calibration_fixed_operating_points.csv",
    tokens=["calibration", "fixed", "operating", "point"],
)

selective_intervals, selective_intervals_path = read_csv_source(
    "notebook07",
    exact_name="Table_selective_operating_point_intervals.csv",
    tokens=["selective", "operating", "interval"],
)

selected_certainty_scores, selected_certainty_scores_path = read_csv_source(
    "notebook07",
    exact_name="02_selected_certainty_scores.csv",
    tokens=["selected", "certainty", "score"],
)

conformal_primary, conformal_primary_path = read_csv_source(
    "notebook08_v2",
    tokens=["primary", "conformal", "result"],
    required=False,
)

if conformal_primary is None:
    conformal_primary, conformal_primary_path = read_csv_source(
        "notebook08_v2",
        tokens=["conformal", "global", "test", "result"],
    )

conformal_classwise, conformal_classwise_path = read_csv_source(
    "notebook08_v2",
    tokens=["conformal", "classwise"],
)

conformal_intervals, conformal_intervals_path = read_csv_source(
    "notebook08_v2",
    tokens=["conformal", "global", "interval"],
    required=False,
)

block_importance, block_importance_path = read_csv_source(
    "notebook09",
    exact_name="Table_block_permutation_importance.csv",
    tokens=["block", "permutation", "importance"],
)

pair_feature_separation, pair_feature_separation_path = read_csv_source(
    "notebook09",
    exact_name="Table_predefined_pair_feature_separation.csv",
    tokens=["pair", "feature", "separation"],
)

pair_block_separation, pair_block_separation_path = read_csv_source(
    "notebook09",
    exact_name="Table_predefined_pair_block_separation.csv",
    tokens=["pair", "block", "separation"],
)

scale_bias, scale_bias_path = read_csv_source(
    "notebook10",
    exact_name="Table_coherent_scale_bias_sensitivity.csv",
    tokens=["coherent", "scale", "bias", "sensitivity"],
)

random_noise_summary, random_noise_summary_path = read_csv_source(
    "notebook10",
    exact_name="Table_random_descriptor_noise_summary.csv",
    tokens=["random", "descriptor", "noise", "summary"],
)

validation_sources = {
    "notebook06": ["final", "inference", "validation"],
    "notebook07": ["final", "selective", "validation"],
    "notebook08_v2": ["final", "conformal", "validation"],
    "notebook09": ["final", "interpretability", "validation"],
    "notebook10": ["final", "measurement", "uncertainty", "validation"],
}

validation_rows = []

for source_key, tokens in validation_sources.items():
    validation_frame, validation_path = read_csv_source(
        source_key,
        tokens=tokens,
    )

    passed_column = first_existing_column(
        validation_frame,
        ["passed", "pass", "valid"],
    )

    validation_rows.append(
        {
            "source": source_key,
            "validation_path": str(validation_path),
            "checks": len(validation_frame),
            "all_passed": bool(
                validation_frame[
                    passed_column
                ].astype(bool).all()
            ),
        }
    )

validation_summary = pd.DataFrame(
    validation_rows
)

if not validation_summary[
    "all_passed"
].all():
    display(validation_summary)
    raise RuntimeError(
        "Algún notebook previo no superó sus validaciones."
    )

display(validation_summary)

In [ ]:
# ============================================================
# 6. Tabla principal 1: conjuntos y particiones
# ============================================================
training_champions = (
    selected_configurations.sort_values(
        [
            "dataset_id",
            "macro_f1_mean",
            "log_loss_mean",
            "candidate_id",
        ],
        ascending=[
            True,
            False,
            True,
            True,
        ],
    )
    .groupby(
        "dataset_id",
        as_index=False,
    )
    .head(1)
    .reset_index(drop=True)
)

split_rows = []

for dataset_id in DATASETS:
    champion_row = training_champions.loc[
        training_champions[
            "dataset_id"
        ] == dataset_id
    ].iloc[0]

    representation = champion_row[
        "representation"
    ]

    candidate_paths = [
        path
        for path in find_parquets(
            extracted_roots["notebook03"]
        )
        if (
            dataset_id in str(path)
            and representation in path.name
        )
    ]

    if not candidate_paths:
        candidate_paths = [
            path
            for path in find_parquets(
                extracted_roots["notebook03"]
            )
            if dataset_id in str(path)
        ]

    if not candidate_paths:
        raise FileNotFoundError(
            f"No se encontró split parquet para {dataset_id}."
        )

    split_path = candidate_paths[0]
    split_frame = pd.read_parquet(split_path)

    for partition in [
        "train",
        "calibration",
        "test",
    ]:
        subset = split_frame.loc[
            split_frame[
                "partition"
            ] == partition
        ]

        split_rows.append(
            {
                "dataset_id": dataset_id,
                "dataset": DATASET_DISPLAY_NAMES[
                    dataset_id
                ],
                "partition": partition,
                "rows": len(subset),
                "groups": subset[
                    "duplicate_group_id"
                ].astype(str).nunique(),
                "classes": subset[
                    "class_label"
                ].astype(str).nunique(),
                "champion_representation": representation,
                "champion_algorithm": champion_row[
                    "algorithm"
                ],
            }
        )

table_1_dataset_design = pd.DataFrame(
    split_rows
)

export_table(
    table_1_dataset_design,
    "Table_1_dataset_design_and_frozen_partitions",
)

display(table_1_dataset_design)

In [ ]:
# ============================================================
# 7. Tabla principal 2: rendimiento de campeones
#    VERSIÓN CORREGIDA PARA FORMATO ANCHO
# ============================================================

# El archivo del Notebook 06 contiene una fila por campeón y
# las métricas distribuidas en columnas:
# test_accuracy, test_accuracy_ci_low, test_accuracy_ci_high, etc.
#
# Esta celda convierte ese formato ancho en una tabla larga
# con una fila por conjunto y métrica.

metric_specification = {
    "accuracy": {
        "estimate": "test_accuracy",
        "ci_low": "test_accuracy_ci_low",
        "ci_high": "test_accuracy_ci_high",
    },
    "balanced_accuracy": {
        "estimate": "test_balanced_accuracy",
        "ci_low": "test_balanced_accuracy_ci_low",
        "ci_high": "test_balanced_accuracy_ci_high",
    },
    "macro_f1": {
        "estimate": "test_macro_f1",
        "ci_low": "test_macro_f1_ci_low",
        "ci_high": "test_macro_f1_ci_high",
    },
    "mcc": {
        "estimate": "test_mcc",
        "ci_low": "test_mcc_ci_low",
        "ci_high": "test_mcc_ci_high",
    },
    "log_loss": {
        "estimate": "test_log_loss",
        "ci_low": "test_log_loss_ci_low",
        "ci_high": "test_log_loss_ci_high",
    },
    "brier": {
        "estimate": "test_brier_multiclass",
        "ci_low": "test_brier_multiclass_ci_low",
        "ci_high": "test_brier_multiclass_ci_high",
    },
}

required_identifier_columns = [
    "dataset_id",
    "representation",
    "algorithm",
]

missing_identifier_columns = [
    column
    for column in required_identifier_columns
    if column not in champion_intervals.columns
]

if missing_identifier_columns:
    raise KeyError(
        "Faltan columnas identificadoras en champion_intervals: "
        f"{missing_identifier_columns}"
    )

required_metric_columns = sorted(
    {
        column_name
        for specification in metric_specification.values()
        for column_name in specification.values()
    }
)

missing_metric_columns = [
    column
    for column in required_metric_columns
    if column not in champion_intervals.columns
]

if missing_metric_columns:
    raise KeyError(
        "Faltan columnas de métricas en champion_intervals: "
        f"{missing_metric_columns}\n"
        f"Columnas disponibles: {list(champion_intervals.columns)}"
    )


table_2_rows = []

for _, champion_row in champion_intervals.iterrows():
    dataset_id = str(
        champion_row["dataset_id"]
    )

    if dataset_id not in DATASET_DISPLAY_NAMES:
        raise KeyError(
            "No existe nombre de presentación para "
            f"dataset_id={dataset_id}"
        )

    for metric_name, specification in metric_specification.items():
        estimate = champion_row[
            specification["estimate"]
        ]

        ci_low = champion_row[
            specification["ci_low"]
        ]

        ci_high = champion_row[
            specification["ci_high"]
        ]

        table_2_rows.append(
            {
                "dataset": DATASET_DISPLAY_NAMES[
                    dataset_id
                ],
                "dataset_id": dataset_id,
                "representation": champion_row[
                    "representation"
                ],
                "algorithm": champion_row[
                    "algorithm"
                ],
                "metric": metric_name,
                "estimate": float(
                    estimate
                ),
                "ci_low": float(
                    ci_low
                ),
                "ci_high": float(
                    ci_high
                ),
            }
        )


table_2_champions = pd.DataFrame(
    table_2_rows
)

expected_rows = (
    champion_intervals[
        "dataset_id"
    ].nunique()
    * len(
        metric_specification
    )
)

if len(table_2_champions) != expected_rows:
    raise RuntimeError(
        "Número inesperado de filas en la tabla de campeones. "
        f"Obtenidas: {len(table_2_champions)}. "
        f"Esperadas: {expected_rows}."
    )

if (
    table_2_champions[
        [
            "estimate",
            "ci_low",
            "ci_high",
        ]
    ]
    .isna()
    .any()
    .any()
):
    display(
        table_2_champions.loc[
            table_2_champions[
                [
                    "estimate",
                    "ci_low",
                    "ci_high",
                ]
            ]
            .isna()
            .any(axis=1)
        ]
    )

    raise RuntimeError(
        "La tabla de campeones contiene valores ausentes."
    )

table_2_champions = (
    table_2_champions.sort_values(
        [
            "dataset_id",
            "metric",
        ]
    )
    .reset_index(
        drop=True
    )
)

export_table(
    table_2_champions,
    "Table_2_training_selected_champion_test_performance",
)

print(
    "Tabla 2 generada correctamente."
)

print(
    "Filas:",
    len(table_2_champions),
)

display(
    table_2_champions
)

In [ ]:
# ============================================================
# 8. Tabla principal 3: ablación tamaño–forma
#    VERSIÓN CORREGIDA
# ============================================================

contrast_metric_column = first_existing_column(
    representation_contrasts,
    ["metric"],
)

contrast_name_column = first_existing_column(
    representation_contrasts,
    ["contrast"],
)

contrast_estimate_column = first_existing_column(
    representation_contrasts,
    [
        "delta_a_minus_b",
        "estimate",
        "delta",
        "mean_difference",
    ],
)

contrast_low_column = first_existing_column(
    representation_contrasts,
    [
        "ci_low",
        "lower_ci",
        "lower",
    ],
)

contrast_high_column = first_existing_column(
    representation_contrasts,
    [
        "ci_high",
        "upper_ci",
        "upper",
    ],
)

holm_column = first_existing_column(
    representation_contrasts,
    [
        "bootstrap_p_holm",
        "p_holm",
        "holm_p",
        "adjusted_p",
    ],
    required=False,
)

reject_column = first_existing_column(
    representation_contrasts,
    [
        "reject_holm_0_05",
        "reject_holm",
        "significant_holm",
    ],
    required=False,
)

required_columns = [
    "dataset_id",
    "algorithm",
    contrast_metric_column,
    contrast_name_column,
    contrast_estimate_column,
    contrast_low_column,
    contrast_high_column,
]

missing_columns = [
    column
    for column in required_columns
    if column not in representation_contrasts.columns
]

if missing_columns:
    raise KeyError(
        "Faltan columnas requeridas en representation_contrasts: "
        f"{missing_columns}\n"
        f"Columnas disponibles: "
        f"{list(representation_contrasts.columns)}"
    )

table_3_ablation = (
    representation_contrasts.loc[
        (
            representation_contrasts[
                "algorithm"
            ] == "rbf_svm"
        )
        & (
            representation_contrasts[
                contrast_metric_column
            ] == "macro_f1"
        )
        & (
            representation_contrasts[
                contrast_name_column
            ].isin(
                [
                    "R1_minus_R3",
                    "R1_minus_R4",
                    "R3_minus_R4",
                ]
            )
        )
    ]
    .copy()
)

table_3_ablation[
    "dataset"
] = table_3_ablation[
    "dataset_id"
].map(
    DATASET_DISPLAY_NAMES
)

if table_3_ablation[
    "dataset"
].isna().any():
    unknown_datasets = sorted(
        table_3_ablation.loc[
            table_3_ablation[
                "dataset"
            ].isna(),
            "dataset_id",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise KeyError(
        "No existe nombre de presentación para: "
        f"{unknown_datasets}"
    )

columns = [
    "dataset",
    "dataset_id",
    "algorithm",
    "representation_a",
    "representation_b",
    contrast_name_column,
    contrast_estimate_column,
    contrast_low_column,
    contrast_high_column,
]

columns = [
    column
    for column in columns
    if column in table_3_ablation.columns
]

if holm_column is not None:
    columns.append(
        holm_column
    )

if reject_column is not None:
    columns.append(
        reject_column
    )

rename_map = {
    contrast_name_column: "contrast",
    contrast_estimate_column: "macro_f1_difference",
    contrast_low_column: "ci_low",
    contrast_high_column: "ci_high",
}

if holm_column is not None:
    rename_map[
        holm_column
    ] = "holm_adjusted_p"

if reject_column is not None:
    rename_map[
        reject_column
    ] = "reject_holm_0_05"

table_3_ablation = (
    table_3_ablation[
        columns
    ]
    .rename(
        columns=rename_map
    )
    .sort_values(
        [
            "dataset_id",
            "contrast",
        ]
    )
    .reset_index(
        drop=True
    )
)

expected_rows = (
    len(DATASETS)
    * 3
)

if len(table_3_ablation) != expected_rows:
    display(
        table_3_ablation
    )

    raise RuntimeError(
        "Número inesperado de contrastes de ablación. "
        f"Obtenidos: {len(table_3_ablation)}. "
        f"Esperados: {expected_rows}."
    )

numeric_columns = [
    "macro_f1_difference",
    "ci_low",
    "ci_high",
]

if table_3_ablation[
    numeric_columns
].isna().any().any():
    raise RuntimeError(
        "La tabla de ablación contiene valores numéricos ausentes."
    )

export_table(
    table_3_ablation,
    "Table_3_size_shape_representation_ablation",
)

print(
    "Tabla 3 generada correctamente."
)

print(
    "Filas:",
    len(table_3_ablation),
)

display(
    table_3_ablation
)

In [ ]:
# ============================================================
# 9. Tabla principal 4: clasificación selectiva
# ============================================================
selected_primary_points = selective_operating_points.loc[
    (
        selective_operating_points[
            "probability_status"
        ] == "temperature_scaled"
    )
    & (
        selective_operating_points[
            "selected_in_calibration"
        ].astype(bool)
    )
    & np.isclose(
        selective_operating_points[
            "target_calibration_coverage"
        ],
        0.90,
    )
].copy()

selected_primary_intervals = selective_intervals.loc[
    (
        selective_intervals[
            "probability_status"
        ] == "temperature_scaled"
    )
    & (
        selective_intervals[
            "selected_in_calibration"
        ].astype(bool)
    )
    & np.isclose(
        selective_intervals[
            "target_calibration_coverage"
        ],
        0.90,
    )
    & (
        selective_intervals[
            "metric"
        ].isin(
            [
                "coverage",
                "selective_accuracy",
                "selective_risk",
                "risk_reduction_vs_no_rejection",
            ]
        )
    )
].copy()

interval_pivot = selected_primary_intervals.pivot_table(
    index=[
        "dataset_id",
        "certainty_score",
    ],
    columns="metric",
    values=[
        "estimate",
        "ci_low",
        "ci_high",
    ],
    aggfunc="first",
)

interval_pivot.columns = [
    f"{metric}_{statistic}"
    for statistic, metric
    in interval_pivot.columns
]

interval_pivot = (
    interval_pivot.reset_index()
)

table_4_selective = (
    selected_primary_points[
        [
            "dataset_id",
            "certainty_score",
            "certainty_threshold",
            "test_n_total",
            "test_n_retained",
        ]
    ]
    .merge(
        interval_pivot,
        on=[
            "dataset_id",
            "certainty_score",
        ],
        how="left",
        validate="one_to_one",
    )
)

table_4_selective["dataset"] = (
    table_4_selective[
        "dataset_id"
    ].map(
        DATASET_DISPLAY_NAMES
    )
)

leading_columns = [
    "dataset",
    "dataset_id",
]

table_4_selective = table_4_selective[
    leading_columns
    + [
        column
        for column in table_4_selective.columns
        if column not in leading_columns
    ]
]

export_table(
    table_4_selective,
    "Table_4_selective_classification_at_90pct_target_coverage",
)

display(table_4_selective)

In [ ]:
# ============================================================
# 10. Tabla principal 5: predicción conformal
# ============================================================
conformal_frame = conformal_primary.copy()

if "probability_status" in conformal_frame.columns:
    conformal_frame = conformal_frame.loc[
        conformal_frame[
            "probability_status"
        ] == "temperature_scaled"
    ]

if "method" in conformal_frame.columns:
    conformal_frame = conformal_frame.loc[
        conformal_frame[
            "method"
        ] == "APS"
    ]

if "calibration_mode" in conformal_frame.columns:
    conformal_frame = conformal_frame.loc[
        conformal_frame[
            "calibration_mode"
        ] == "marginal"
    ]

conformal_columns = [
    column
    for column in [
        "dataset_id",
        "representation",
        "algorithm",
        "nominal_coverage",
        "coverage",
        "coverage_gap",
        "mean_set_size",
        "median_set_size",
        "singleton_rate",
        "empty_set_rate",
        "minimum_class_coverage",
        "class_coverage_range",
    ]
    if column in conformal_frame.columns
]

table_5_conformal = conformal_frame[
    conformal_columns
].copy()

table_5_conformal["dataset"] = (
    table_5_conformal[
        "dataset_id"
    ].map(
        DATASET_DISPLAY_NAMES
    )
)

table_5_conformal = table_5_conformal[
    [
        "dataset",
        *[
            column
            for column in table_5_conformal.columns
            if column != "dataset"
        ],
    ]
]

export_table(
    table_5_conformal,
    "Table_5_canonical_APS_marginal_conformal_prediction",
)

display(table_5_conformal)

In [ ]:
# ============================================================
# 11. Main Tables 6, 7A and 7B
#     Morphometric interpretation and measurement sensitivity
# ============================================================
BLOCK_DISPLAY_NAMES = {
    "size_dependent": "Size-dependent descriptors",
    "invariant_shape": "Invariant-shape descriptors",
    "size_anchor": "Size anchor (log-area)",
    "all_features": "All descriptors",
}

# ------------------------------------------------------------
# Table 6. Grouped permutation importance
# ------------------------------------------------------------
table_6_block_importance = block_importance.loc[
    block_importance[
        "partition"
    ] == "test"
].copy()

table_6_block_importance["dataset"] = (
    table_6_block_importance[
        "dataset_id"
    ].map(
        DATASET_DISPLAY_NAMES
    )
)

table_6_block_importance["block_label"] = (
    table_6_block_importance[
        "block"
    ].map(
        BLOCK_DISPLAY_NAMES
    )
)

if table_6_block_importance[
    "block_label"
].isna().any():
    unknown_blocks = sorted(
        table_6_block_importance.loc[
            table_6_block_importance[
                "block_label"
            ].isna(),
            "block",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise KeyError(
        "Missing publication label for blocks: "
        f"{unknown_blocks}"
    )

table_6_block_importance = (
    table_6_block_importance[
        [
            "dataset",
            "dataset_id",
            "representation",
            "algorithm",
            "block",
            "block_label",
            "n_features",
            "importance_mean",
            "importance_sd",
            "interval_low",
            "interval_high",
        ]
    ]
    .sort_values(
        [
            "dataset_id",
            "importance_mean",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

export_table(
    table_6_block_importance,
    "Table_6_grouped_morphometric_permutation_importance",
)

# ------------------------------------------------------------
# Table 7A. Coherent scale-bias sensitivity
# ------------------------------------------------------------
table_7A_scale_sensitivity = scale_bias.loc[
    scale_bias[
        "scale_bias_percent"
    ].isin(
        [
            -5.0,
            -2.0,
            0.0,
            2.0,
            5.0,
        ]
    )
].copy()

table_7A_scale_sensitivity["dataset"] = (
    table_7A_scale_sensitivity[
        "dataset_id"
    ].map(
        DATASET_DISPLAY_NAMES
    )
)

table_7A_scale_sensitivity = (
    table_7A_scale_sensitivity[
        [
            "dataset",
            "dataset_id",
            "scale_bias_percent",
            "macro_f1",
            "macro_f1_drop",
            "label_flip_rate",
            "mean_absolute_probability_shift",
            "selective_coverage",
            "selective_accuracy",
            "selective_risk",
            "selective_risk_increase",
        ]
    ]
    .sort_values(
        [
            "dataset_id",
            "scale_bias_percent",
        ]
    )
    .reset_index(drop=True)
)

export_table(
    table_7A_scale_sensitivity,
    "Table_7A_coherent_scale_bias_sensitivity",
)

# ------------------------------------------------------------
# Table 7B. Random descriptor-noise sensitivity
# ------------------------------------------------------------
table_7B_random_noise = random_noise_summary.loc[
    (
        random_noise_summary[
            "scenario"
        ] == "all_features"
    )
    & (
        random_noise_summary[
            "metric"
        ].isin(
            [
                "macro_f1",
                "macro_f1_drop",
                "label_flip_rate",
                "selective_coverage",
                "selective_risk",
            ]
        )
    )
    & (
        random_noise_summary[
            "noise_level_percent_robust_scale"
        ].isin(
            [
                5.0,
                10.0,
            ]
        )
    )
].copy()

table_7B_random_noise["dataset"] = (
    table_7B_random_noise[
        "dataset_id"
    ].map(
        DATASET_DISPLAY_NAMES
    )
)

table_7B_random_noise["scenario_label"] = (
    table_7B_random_noise[
        "scenario"
    ].map(
        BLOCK_DISPLAY_NAMES
    )
)

table_7B_random_noise = (
    table_7B_random_noise[
        [
            "dataset",
            "dataset_id",
            "scenario",
            "scenario_label",
            "noise_level_percent_robust_scale",
            "metric",
            "mean",
            "sd",
            "interval_low",
            "interval_high",
            "minimum",
            "maximum",
            "n_replicates",
        ]
    ]
    .sort_values(
        [
            "dataset_id",
            "noise_level_percent_robust_scale",
            "metric",
        ]
    )
    .reset_index(drop=True)
)

export_table(
    table_7B_random_noise,
    "Table_7B_random_descriptor_noise_sensitivity",
)

# Preserve focused supplementary extracts.
table_7A_scale_sensitivity.to_csv(
    SUPPLEMENT_DIR
    / "Supplementary_scale_bias_selected_points.csv",
    index=False,
    encoding="utf-8-sig",
)

table_7B_random_noise.to_csv(
    SUPPLEMENT_DIR
    / "Supplementary_random_noise_selected_points.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Tables 6, 7A and 7B generated successfully."
)

display(
    table_6_block_importance
)

display(
    table_7A_scale_sensitivity
)

display(
    table_7B_random_noise
)

In [ ]:
# ============================================================
# 12. Main figures — publication-ready English labels
# ============================================================
REPRESENTATION_DISPLAY_NAMES = {
    "R1_full_native": "R1: Full native",
    "R2_measured_core": "R2: Measured core",
    "R3_invariant_core": "R3: Invariant shape",
    "R4_hybrid_log_area": "R4: Shape + log-area",
}

# ------------------------------------------------------------
# Figure 1. Macro-F1 by representation for the RBF-SVM
# ------------------------------------------------------------
representation_plot = model_metric_intervals.loc[
    (
        model_metric_intervals[
            "algorithm"
        ] == "rbf_svm"
    )
    & (
        model_metric_intervals[
            "probability_status"
        ] == "native"
    )
    & (
        model_metric_intervals[
            "metric"
        ] == "macro_f1"
    )
].copy()

representation_plot["representation_order"] = (
    representation_plot[
        "representation"
    ].map(
        {
            representation: index
            for index, representation
            in enumerate(
                REPRESENTATION_ORDER
            )
        }
    )
)

representation_plot["representation_label"] = (
    representation_plot[
        "representation"
    ].map(
        REPRESENTATION_DISPLAY_NAMES
    )
)

if representation_plot[
    "representation_label"
].isna().any():
    raise KeyError(
        "A representation is missing its publication label."
    )

figure, axis = plt.subplots(
    figsize=(10, 6)
)

x_positions = np.arange(
    len(
        REPRESENTATION_ORDER
    )
)

for dataset_id, subset in representation_plot.groupby(
    "dataset_id",
    sort=True,
):
    subset = (
        subset.sort_values(
            "representation_order"
        )
        .reset_index(drop=True)
    )

    axis.errorbar(
        subset[
            "representation_order"
        ],
        subset[
            "estimate"
        ],
        yerr=[
            subset[
                "estimate"
            ]
            - subset[
                "ci_low"
            ],
            subset[
                "ci_high"
            ]
            - subset[
                "estimate"
            ],
        ],
        marker="o",
        capsize=4,
        label=DATASET_DISPLAY_NAMES[
            dataset_id
        ],
    )

axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    [
        REPRESENTATION_DISPLAY_NAMES[
            representation
        ]
        for representation in REPRESENTATION_ORDER
    ],
    rotation=20,
    ha="right",
)

axis.set_ylabel(
    "Macro-F1"
)

axis.set_xlabel(
    "Morphometric representation"
)

axis.set_title(
    "RBF-SVM performance across morphometric representations"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

figure.tight_layout()

save_figure(
    figure,
    "Figure_1_representation_macro_f1",
)

plt.show()


# ------------------------------------------------------------
# Figure 2. Selective risk versus empirical coverage
# ------------------------------------------------------------
figure, axis = plt.subplots(
    figsize=(8, 6)
)

selected_curve_points = selective_operating_points.loc[
    (
        selective_operating_points[
            "probability_status"
        ] == "temperature_scaled"
    )
    & (
        selective_operating_points[
            "selected_in_calibration"
        ].astype(bool)
    )
].copy()

for dataset_id, subset in selected_curve_points.groupby(
    "dataset_id",
    sort=True,
):
    subset = subset.sort_values(
        "test_coverage"
    )

    axis.plot(
        subset[
            "test_coverage"
        ],
        subset[
            "test_selective_risk"
        ],
        marker="o",
        label=DATASET_DISPLAY_NAMES[
            dataset_id
        ],
    )

axis.set_xlabel(
    "Empirical coverage"
)

axis.set_ylabel(
    "Selective risk"
)

axis.set_title(
    "Selective classification using calibration-defined thresholds"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

figure.tight_layout()

save_figure(
    figure,
    "Figure_2_selective_risk_coverage",
)

plt.show()


# ------------------------------------------------------------
# Figure 3. Conformal coverage and set efficiency
# ------------------------------------------------------------
figure, axis = plt.subplots(
    figsize=(8, 6)
)

for dataset_id, subset in table_5_conformal.groupby(
    "dataset_id",
    sort=True,
):
    subset = subset.sort_values(
        "nominal_coverage"
    )

    axis.plot(
        subset[
            "mean_set_size"
        ],
        subset[
            "coverage"
        ],
        marker="o",
        label=DATASET_DISPLAY_NAMES[
            dataset_id
        ],
    )

    for _, row in subset.iterrows():
        axis.annotate(
            f"{int(100 * row['nominal_coverage'])}%",
            (
                row[
                    "mean_set_size"
                ],
                row[
                    "coverage"
                ],
            ),
            xytext=(5, 5),
            textcoords="offset points",
        )

axis.set_xlabel(
    "Mean prediction-set size"
)

axis.set_ylabel(
    "Empirical coverage"
)

axis.set_title(
    "Coverage–efficiency trade-off for canonical marginal APS"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

figure.tight_layout()

save_figure(
    figure,
    "Figure_3_conformal_coverage_efficiency",
)

plt.show()


# ------------------------------------------------------------
# Figure 4. Grouped morphometric importance
# ------------------------------------------------------------
figure, axis = plt.subplots(
    figsize=(9, 6)
)

plot_table = (
    table_6_block_importance.sort_values(
        [
            "dataset_id",
            "importance_mean",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

plot_positions = np.arange(
    len(
        plot_table
    )
)

plot_labels = (
    plot_table[
        "dataset"
    ]
    + " — "
    + plot_table[
        "block_label"
    ]
)

axis.barh(
    plot_positions,
    plot_table[
        "importance_mean"
    ],
    xerr=plot_table[
        "importance_sd"
    ],
)

axis.set_yticks(
    plot_positions
)

axis.set_yticklabels(
    plot_labels
)

axis.set_xlabel(
    "Mean decrease in macro-F1"
)

axis.set_title(
    "Grouped permutation importance of morphometric descriptors"
)

axis.grid(
    True,
    axis="x",
    alpha=0.3,
)

figure.tight_layout()

save_figure(
    figure,
    "Figure_4_grouped_morphometric_importance",
)

plt.show()


# ------------------------------------------------------------
# Figure 5. Coherent scale-bias sensitivity
# ------------------------------------------------------------
figure, axis = plt.subplots(
    figsize=(8, 6)
)

for dataset_id, subset in scale_bias.groupby(
    "dataset_id",
    sort=True,
):
    subset = subset.sort_values(
        "scale_bias_percent"
    )

    axis.plot(
        subset[
            "scale_bias_percent"
        ],
        subset[
            "macro_f1"
        ],
        marker="o",
        label=DATASET_DISPLAY_NAMES[
            dataset_id
        ],
    )

axis.set_xlabel(
    "Coherent scale bias (%)"
)

axis.set_ylabel(
    "Macro-F1"
)

axis.set_title(
    "Sensitivity to systematic scale error"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

figure.tight_layout()

save_figure(
    figure,
    "Figure_5_coherent_scale_bias",
)

plt.show()


# ------------------------------------------------------------
# Figure 6. Random descriptor-noise sensitivity
# ------------------------------------------------------------
figure, axis = plt.subplots(
    figsize=(8, 6)
)

noise_plot = random_noise_summary.loc[
    (
        random_noise_summary[
            "scenario"
        ] == "all_features"
    )
    & (
        random_noise_summary[
            "metric"
        ] == "macro_f1_drop"
    )
].copy()

for dataset_id, subset in noise_plot.groupby(
    "dataset_id",
    sort=True,
):
    subset = subset.sort_values(
        "noise_level_percent_robust_scale"
    )

    axis.plot(
        subset[
            "noise_level_percent_robust_scale"
        ],
        subset[
            "mean"
        ],
        marker="o",
        label=DATASET_DISPLAY_NAMES[
            dataset_id
        ],
    )

    axis.fill_between(
        subset[
            "noise_level_percent_robust_scale"
        ],
        subset[
            "interval_low"
        ],
        subset[
            "interval_high"
        ],
        alpha=0.15,
    )

axis.axhline(
    0.02,
    linestyle="--",
    linewidth=1,
    label="Prespecified practical-loss threshold",
)

axis.set_xlabel(
    "Noise magnitude (% of the training robust scale)"
)

axis.set_ylabel(
    "Macro-F1 decrease"
)

axis.set_title(
    "Sensitivity to random descriptor noise"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

figure.tight_layout()

save_figure(
    figure,
    "Figure_6_random_descriptor_noise",
)

plt.show()

In [ ]:
# ============================================================
# 13. Material suplementario e índice de procedencia
# ============================================================
source_tables = {
    "S01_selected_model_configurations": selected_config_path,
    "S02_champion_test_intervals": champion_intervals_path,
    "S03_representation_contrasts": representation_contrasts_path,
    "S04_model_metric_intervals": model_metric_intervals_path,
    "S05_selective_operating_points": selective_operating_points_path,
    "S06_selective_intervals": selective_intervals_path,
    "S07_selected_certainty_scores": selected_certainty_scores_path,
    "S08_conformal_primary": conformal_primary_path,
    "S09_conformal_classwise": conformal_classwise_path,
    "S10_block_importance": block_importance_path,
    "S11_pair_feature_separation": pair_feature_separation_path,
    "S12_pair_block_separation": pair_block_separation_path,
    "S13_scale_bias": scale_bias_path,
    "S14_random_noise_summary": random_noise_summary_path,
}

provenance_rows = []

for table_id, source_path in source_tables.items():
    copied_path = copy_source_table(
        source_path,
        table_id,
    )

    provenance_rows.append(
        {
            "asset_id": table_id,
            "source_path": str(
                source_path
            ),
            "copied_path": str(
                copied_path.relative_to(
                    OUTPUT_ROOT
                )
            ),
            "source_notebook": next(
                (
                    key
                    for key, root
                    in extracted_roots.items()
                    if root in source_path.parents
                ),
                "unknown",
            ),
        }
    )

provenance_index = pd.DataFrame(
    provenance_rows
)

provenance_index.to_csv(
    RESULTS_DIR
    / "01_source_provenance_index.csv",
    index=False,
    encoding="utf-8-sig",
)

validation_summary.to_csv(
    RESULTS_DIR
    / "02_prior_notebook_validation_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

display(provenance_index)

In [ ]:
# ============================================================
# 14. Traceable numerical claim register
# ============================================================
claim_rows = []

# ------------------------------------------------------------
# Predictive performance
# ------------------------------------------------------------
for _, row in table_2_champions.iterrows():
    claim_rows.append(
        {
            "claim_id": (
                f"CHAMPION_"
                f"{row['dataset_id']}_"
                f"{row['metric']}"
            ),
            "section": "Predictive performance",
            "dataset_id": row["dataset_id"],
            "claim_type": "frozen_test_metric",
            "claim_text": (
                f"{row['dataset']} champion "
                f"{row['metric']} = "
                f"{row['estimate']:.4f} "
                f"(95% interval "
                f"{row['ci_low']:.4f}–"
                f"{row['ci_high']:.4f})."
            ),
            "source_asset": (
                "Table_2_training_selected_champion_test_performance"
            ),
            "source_notebook": "notebook06",
        }
    )

# ------------------------------------------------------------
# Size–shape ablation
# ------------------------------------------------------------
for _, row in table_3_ablation.iterrows():
    significance_text = ""

    if "reject_holm_0_05" in row.index:
        significance_text = (
            " The Holm-adjusted comparison "
            + (
                "was significant."
                if bool(
                    row[
                        "reject_holm_0_05"
                    ]
                )
                else "was not significant."
            )
        )

    claim_rows.append(
        {
            "claim_id": (
                f"ABLATION_"
                f"{row['dataset_id']}_"
                f"{row['contrast']}"
            ),
            "section": "Size–shape ablation",
            "dataset_id": row["dataset_id"],
            "claim_type": (
                "paired_representation_contrast"
            ),
            "claim_text": (
                f"{row['dataset']} "
                f"{row['contrast']} macro-F1 "
                f"difference = "
                f"{row['macro_f1_difference']:.4f} "
                f"(95% interval "
                f"{row['ci_low']:.4f}–"
                f"{row['ci_high']:.4f})."
                f"{significance_text}"
            ),
            "source_asset": (
                "Table_3_size_shape_representation_ablation"
            ),
            "source_notebook": "notebook06",
        }
    )

# ------------------------------------------------------------
# Selective classification
# ------------------------------------------------------------
for _, row in table_4_selective.iterrows():
    claim_rows.append(
        {
            "claim_id": (
                f"SELECTIVE_"
                f"{row['dataset_id']}_90"
            ),
            "section": "Selective classification",
            "dataset_id": row["dataset_id"],
            "claim_type": (
                "calibration_fixed_operating_point"
            ),
            "claim_text": (
                f"{row['dataset']} achieved "
                f"coverage "
                f"{row['coverage_estimate']:.4f}, "
                f"selective accuracy "
                f"{row['selective_accuracy_estimate']:.4f}, "
                f"and selective risk "
                f"{row['selective_risk_estimate']:.4f} "
                f"at the calibration-defined "
                f"90% target operating point."
            ),
            "source_asset": (
                "Table_4_selective_classification_at_90pct_target_coverage"
            ),
            "source_notebook": "notebook07",
        }
    )

# ------------------------------------------------------------
# Conformal prediction
# ------------------------------------------------------------
for _, row in table_5_conformal.iterrows():
    claim_rows.append(
        {
            "claim_id": (
                f"CONFORMAL_"
                f"{row['dataset_id']}_"
                f"{int(100 * row['nominal_coverage'])}"
            ),
            "section": "Conformal prediction",
            "dataset_id": row["dataset_id"],
            "claim_type": (
                "canonical_APS_marginal"
            ),
            "claim_text": (
                f"{row['dataset']} canonical marginal APS "
                f"coverage = {row['coverage']:.4f} "
                f"at nominal coverage "
                f"{row['nominal_coverage']:.2f}, "
                f"with mean set size "
                f"{row['mean_set_size']:.4f}."
            ),
            "source_asset": (
                "Table_5_canonical_APS_marginal_conformal_prediction"
            ),
            "source_notebook": "notebook08_v2",
        }
    )

# ------------------------------------------------------------
# Grouped morphometric interpretation
# ------------------------------------------------------------
for _, row in table_6_block_importance.iterrows():
    claim_rows.append(
        {
            "claim_id": (
                f"IMPORTANCE_"
                f"{row['dataset_id']}_"
                f"{row['block']}"
            ),
            "section": "Morphometric interpretation",
            "dataset_id": row["dataset_id"],
            "claim_type": (
                "grouped_permutation_importance"
            ),
            "claim_text": (
                f"{row['dataset']} "
                f"{row['block_label']} produced a "
                f"mean macro-F1 decrease of "
                f"{row['importance_mean']:.4f} "
                f"across test-set permutations "
                f"(permutation interval "
                f"{row['interval_low']:.4f}–"
                f"{row['interval_high']:.4f})."
            ),
            "source_asset": (
                "Table_6_grouped_morphometric_permutation_importance"
            ),
            "source_notebook": "notebook09",
        }
    )

# ------------------------------------------------------------
# Predefined difficult-pair interpretation
# ------------------------------------------------------------
for dataset_id in DATASETS:
    dataset_name = DATASET_DISPLAY_NAMES[
        dataset_id
    ]

    feature_subset = (
        pair_feature_separation.loc[
            pair_feature_separation[
                "dataset_id"
            ] == dataset_id
        ]
        .sort_values(
            "univariate_auc_direction_adjusted",
            ascending=False,
        )
    )

    block_subset = (
        pair_block_separation.loc[
            pair_block_separation[
                "dataset_id"
            ] == dataset_id
        ]
        .sort_values(
            "euclidean_centroid_distance",
            ascending=False,
        )
    )

    if feature_subset.empty or block_subset.empty:
        raise RuntimeError(
            f"Missing predefined-pair results for {dataset_id}."
        )

    feature_row = feature_subset.iloc[0]
    block_row = block_subset.iloc[0]

    claim_rows.append(
        {
            "claim_id": (
                f"PAIR_FEATURE_"
                f"{dataset_id}"
            ),
            "section": "Morphometric interpretation",
            "dataset_id": dataset_id,
            "claim_type": (
                "predefined_pair_univariate_separation"
            ),
            "claim_text": (
                f"For the predefined "
                f"{feature_row['class_a']}–"
                f"{feature_row['class_b']} pair in "
                f"{dataset_name}, "
                f"{feature_row['feature']} had the "
                f"highest direction-adjusted univariate AUC "
                f"({feature_row['univariate_auc_direction_adjusted']:.4f})."
            ),
            "source_asset": (
                "S11_pair_feature_separation"
            ),
            "source_notebook": "notebook09",
        }
    )

    claim_rows.append(
        {
            "claim_id": (
                f"PAIR_BLOCK_"
                f"{dataset_id}"
            ),
            "section": "Morphometric interpretation",
            "dataset_id": dataset_id,
            "claim_type": (
                "predefined_pair_centroid_separation"
            ),
            "claim_text": (
                f"For the predefined "
                f"{block_row['class_a']}–"
                f"{block_row['class_b']} pair in "
                f"{dataset_name}, the "
                f"{BLOCK_DISPLAY_NAMES.get(block_row['block'], block_row['block'])} "
                f"block had the largest centroid distance "
                f"({block_row['euclidean_centroid_distance']:.4f})."
            ),
            "source_asset": (
                "S12_pair_block_separation"
            ),
            "source_notebook": "notebook09",
        }
    )

# ------------------------------------------------------------
# Coherent scale-bias sensitivity at ±5%
# ------------------------------------------------------------
scale_claim_rows = table_7A_scale_sensitivity.loc[
    table_7A_scale_sensitivity[
        "scale_bias_percent"
    ].isin(
        [
            -5.0,
            5.0,
        ]
    )
]

for _, row in scale_claim_rows.iterrows():
    direction_label = (
        "minus5"
        if row[
            "scale_bias_percent"
        ] < 0
        else "plus5"
    )

    claim_rows.append(
        {
            "claim_id": (
                f"SCALE_"
                f"{row['dataset_id']}_"
                f"{direction_label}"
            ),
            "section": "Measurement uncertainty",
            "dataset_id": row["dataset_id"],
            "claim_type": (
                "coherent_scale_bias_sensitivity"
            ),
            "claim_text": (
                f"Under a coherent scale bias of "
                f"{row['scale_bias_percent']:+.0f}%, "
                f"{row['dataset']} macro-F1 was "
                f"{row['macro_f1']:.4f}, corresponding to "
                f"a baseline decrease of "
                f"{row['macro_f1_drop']:.4f} and a "
                f"label-flip rate of "
                f"{row['label_flip_rate']:.4f}."
            ),
            "source_asset": (
                "Table_7A_coherent_scale_bias_sensitivity"
            ),
            "source_notebook": "notebook10",
        }
    )

# ------------------------------------------------------------
# Random descriptor-noise sensitivity at 10%
# ------------------------------------------------------------
noise_claim_rows = table_7B_random_noise.loc[
    (
        table_7B_random_noise[
            "noise_level_percent_robust_scale"
        ] == 10.0
    )
    & (
        table_7B_random_noise[
            "metric"
        ].isin(
            [
                "macro_f1_drop",
                "selective_coverage",
                "selective_risk",
            ]
        )
    )
]

for _, row in noise_claim_rows.iterrows():
    claim_rows.append(
        {
            "claim_id": (
                f"NOISE_"
                f"{row['dataset_id']}_10_"
                f"{row['metric']}"
            ),
            "section": "Measurement uncertainty",
            "dataset_id": row["dataset_id"],
            "claim_type": (
                "random_descriptor_noise_sensitivity"
            ),
            "claim_text": (
                f"With random noise equal to 10% of the "
                f"training robust scale across all descriptors, "
                f"{row['dataset']} "
                f"{row['metric']} averaged "
                f"{row['mean']:.4f} "
                f"(Monte Carlo interval "
                f"{row['interval_low']:.4f}–"
                f"{row['interval_high']:.4f})."
            ),
            "source_asset": (
                "Table_7B_random_descriptor_noise_sensitivity"
            ),
            "source_notebook": "notebook10",
        }
    )

claim_register = (
    pd.DataFrame(
        claim_rows
    )
    .sort_values(
        [
            "section",
            "dataset_id",
            "claim_id",
        ]
    )
    .reset_index(drop=True)
)

if claim_register[
    "claim_id"
].duplicated().any():
    duplicates = claim_register.loc[
        claim_register[
            "claim_id"
        ].duplicated(
            keep=False
        ),
        "claim_id",
    ].tolist()

    raise RuntimeError(
        "Duplicate claim identifiers: "
        f"{duplicates}"
    )

claim_register.to_csv(
    RESULTS_DIR
    / "03_claim_register.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Claim register entries:",
    len(
        claim_register
    ),
)

display(
    claim_register
)

In [ ]:
# ============================================================
# 15. English numerical summary for Results drafting
# ============================================================
def champion_value(
    dataset_id,
    metric,
):
    row = table_2_champions.loc[
        (
            table_2_champions[
                "dataset_id"
            ] == dataset_id
        )
        & (
            table_2_champions[
                "metric"
            ] == metric
        )
    ].iloc[0]

    return (
        row["estimate"],
        row["ci_low"],
        row["ci_high"],
    )

summary_lines = [
    "# Numeric Results Summary",
    "",
    "This file is generated from frozen notebook outputs.",
    "Each numerical statement is linked to a source table through the claim register.",
    "",
]

for dataset_id in DATASETS:
    dataset_name = DATASET_DISPLAY_NAMES[
        dataset_id
    ]

    macro_value, macro_low, macro_high = champion_value(
        dataset_id,
        "macro_f1",
    )

    accuracy_value, accuracy_low, accuracy_high = champion_value(
        dataset_id,
        "accuracy",
    )

    summary_lines.extend(
        [
            f"## {dataset_name}",
            "",
            (
                f"- Champion test accuracy: "
                f"{accuracy_value:.4f} "
                f"(95% interval "
                f"{accuracy_low:.4f}–"
                f"{accuracy_high:.4f})."
            ),
            (
                f"- Champion test macro-F1: "
                f"{macro_value:.4f} "
                f"(95% interval "
                f"{macro_low:.4f}–"
                f"{macro_high:.4f})."
            ),
        ]
    )

    selective_row = table_4_selective.loc[
        table_4_selective[
            "dataset_id"
        ] == dataset_id
    ].iloc[0]

    summary_lines.append(
        (
            f"- Selective classification at the "
            f"calibration-defined 90% target: "
            f"coverage "
            f"{selective_row['coverage_estimate']:.4f}, "
            f"selective accuracy "
            f"{selective_row['selective_accuracy_estimate']:.4f}, "
            f"and selective risk "
            f"{selective_row['selective_risk_estimate']:.4f}."
        )
    )

    conformal_subset = table_5_conformal.loc[
        table_5_conformal[
            "dataset_id"
        ] == dataset_id
    ].sort_values(
        "nominal_coverage"
    )

    for _, conformal_row in conformal_subset.iterrows():
        summary_lines.append(
            (
                f"- Canonical marginal APS at nominal "
                f"{100 * conformal_row['nominal_coverage']:.0f}%: "
                f"coverage "
                f"{conformal_row['coverage']:.4f}, "
                f"mean set size "
                f"{conformal_row['mean_set_size']:.4f}, "
                f"and singleton rate "
                f"{conformal_row['singleton_rate']:.4f}."
            )
        )

    importance_subset = (
        table_6_block_importance.loc[
            table_6_block_importance[
                "dataset_id"
            ] == dataset_id
        ]
        .sort_values(
            "importance_mean",
            ascending=False,
        )
    )

    for _, importance_row in importance_subset.iterrows():
        summary_lines.append(
            (
                f"- Grouped permutation importance for "
                f"{importance_row['block_label']}: "
                f"mean macro-F1 decrease "
                f"{importance_row['importance_mean']:.4f}."
            )
        )

    scale_subset = table_7A_scale_sensitivity.loc[
        (
            table_7A_scale_sensitivity[
                "dataset_id"
            ] == dataset_id
        )
        & (
            table_7A_scale_sensitivity[
                "scale_bias_percent"
            ].isin(
                [
                    -5.0,
                    5.0,
                ]
            )
        )
    ].sort_values(
        "scale_bias_percent"
    )

    for _, scale_row in scale_subset.iterrows():
        summary_lines.append(
            (
                f"- At coherent scale bias "
                f"{scale_row['scale_bias_percent']:+.0f}%, "
                f"macro-F1 was "
                f"{scale_row['macro_f1']:.4f}, with a "
                f"baseline decrease of "
                f"{scale_row['macro_f1_drop']:.4f}."
            )
        )

    noise_row = table_7B_random_noise.loc[
        (
            table_7B_random_noise[
                "dataset_id"
            ] == dataset_id
        )
        & (
            table_7B_random_noise[
                "noise_level_percent_robust_scale"
            ] == 10.0
        )
        & (
            table_7B_random_noise[
                "metric"
            ] == "macro_f1_drop"
        )
    ].iloc[0]

    summary_lines.append(
        (
            f"- With random noise equal to 10% of the "
            f"training robust scale across all descriptors, "
            f"the mean macro-F1 decrease was "
            f"{noise_row['mean']:.4f} "
            f"(Monte Carlo interval "
            f"{noise_row['interval_low']:.4f}–"
            f"{noise_row['interval_high']:.4f})."
        )
    )

    summary_lines.append("")

(TEXT_DIR / "Numeric_results_summary.md").write_text(
    "\n".join(
        summary_lines
    ),
    encoding="utf-8",
)

print(
    "\n".join(
        summary_lines[:35]
    )
)

In [ ]:
# ============================================================
# 16. Final validation and integrity manifest
# ============================================================
main_table_names = [
    "Table_1_dataset_design_and_frozen_partitions",
    "Table_2_training_selected_champion_test_performance",
    "Table_3_size_shape_representation_ablation",
    "Table_4_selective_classification_at_90pct_target_coverage",
    "Table_5_canonical_APS_marginal_conformal_prediction",
    "Table_6_grouped_morphometric_permutation_importance",
    "Table_7A_coherent_scale_bias_sensitivity",
    "Table_7B_random_descriptor_noise_sensitivity",
]

main_figure_names = [
    "Figure_1_representation_macro_f1",
    "Figure_2_selective_risk_coverage",
    "Figure_3_conformal_coverage_efficiency",
    "Figure_4_grouped_morphometric_importance",
    "Figure_5_coherent_scale_bias",
    "Figure_6_random_descriptor_noise",
]

expected_claim_count = (
    len(
        table_2_champions
    )
    + len(
        table_3_ablation
    )
    + len(
        table_4_selective
    )
    + len(
        table_5_conformal
    )
    + len(
        table_6_block_importance
    )
    + 2
    * len(
        DATASETS
    )
    + len(
        scale_claim_rows
    )
    + len(
        noise_claim_rows
    )
)

validation_checks = {
    "all_prior_notebook_validations_passed": bool(
        validation_summary[
            "all_passed"
        ].all()
    ),
    "exactly_two_training_selected_champions": (
        len(
            training_champions
        )
        == len(
            DATASETS
        )
    ),
    "table_1_contains_all_partitions": (
        len(
            table_1_dataset_design
        )
        == len(
            DATASETS
        )
        * 3
    ),
    "table_2_contains_six_metrics_per_dataset": (
        len(
            table_2_champions
        )
        == len(
            DATASETS
        )
        * 6
    ),
    "table_3_contains_six_svm_contrasts": (
        len(
            table_3_ablation
        )
        == len(
            DATASETS
        )
        * 3
    ),
    "table_4_contains_two_primary_operating_points": (
        len(
            table_4_selective
        )
        == len(
            DATASETS
        )
    ),
    "table_5_contains_two_nominal_levels_per_dataset": (
        len(
            table_5_conformal
        )
        == len(
            DATASETS
        )
        * 2
    ),
    "table_6_contains_test_block_importance": (
        table_6_block_importance[
            "dataset_id"
        ].nunique()
        == len(
            DATASETS
        )
    ),
    "table_7A_contains_five_scale_points_per_dataset": (
        len(
            table_7A_scale_sensitivity
        )
        == len(
            DATASETS
        )
        * 5
    ),
    "table_7B_contains_two_noise_levels_and_five_metrics": (
        len(
            table_7B_random_noise
        )
        == len(
            DATASETS
        )
        * 2
        * 5
    ),
    "claim_register_has_expected_entries": (
        len(
            claim_register
        )
        == expected_claim_count
    ),
    "claim_register_contains_interpretability": (
        (
            claim_register[
                "section"
            ]
            == "Morphometric interpretation"
        ).any()
    ),
    "claim_register_contains_measurement_uncertainty": (
        (
            claim_register[
                "section"
            ]
            == "Measurement uncertainty"
        ).any()
    ),
    "claim_ids_are_unique": (
        not claim_register[
            "claim_id"
        ].duplicated().any()
    ),
    "all_main_table_formats_exist": all(
        (
            TABLES_DIR
            / f"{name}.csv"
        ).exists()
        and (
            TABLES_DIR
            / f"{name}.md"
        ).exists()
        and (
            TABLES_DIR
            / f"{name}.tex"
        ).exists()
        for name in main_table_names
    ),
    "all_main_figures_exist_in_png_and_pdf": all(
        (
            FIGURES_DIR
            / f"{name}.png"
        ).exists()
        and (
            FIGURES_DIR
            / f"{name}.pdf"
        ).exists()
        for name in main_figure_names
    ),
    "all_main_figure_labels_are_english": True,
    "no_model_training_or_reselection": True,
}

final_validation = pd.DataFrame(
    [
        {
            "check": check,
            "passed": passed,
        }
        for check, passed
        in validation_checks.items()
    ]
)

final_validation.to_csv(
    RESULTS_DIR
    / "04_final_manuscript_assets_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

if not final_validation[
    "passed"
].astype(bool).all():
    display(
        final_validation
    )

    raise RuntimeError(
        "At least one final validation failed."
    )

protocol = {
    "protocol_name": (
        "Grain_AI_MDPI_manuscript_assets_protocol_v2_editorial"
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "source_notebooks": list(
        ZIP_FILENAMES.keys()
    ),
    "main_tables": (
        main_table_names
    ),
    "main_figures": (
        main_figure_names
    ),
    "figure_language": "English",
    "claim_register_entries": int(
        len(
            claim_register
        )
    ),
    "restrictions": [
        "No model training.",
        "No hyperparameter selection.",
        "No probability recalibration.",
        "No threshold recalibration.",
        "No test-based reselection.",
        "All numerical claims must appear in the claim register or source tables.",
        "Permutation intervals are not sampling confidence intervals.",
        "Monte Carlo intervals describe perturbation variability.",
        "No image-level robustness claim.",
    ],
}

(
    PROTOCOL_DIR
    / "manuscript_assets_protocol_v2_editorial.json"
).write_text(
    json.dumps(
        protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

def sha256_file(
    path: Path,
):
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(
                1024
                * 1024
            ),
            b"",
        ):
            digest.update(
                block
            )

    return digest.hexdigest()

manifest_rows = []

for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if path.is_file():
        manifest_rows.append(
            {
                "relative_path": str(
                    path.relative_to(
                        OUTPUT_ROOT
                    )
                ),
                "size_bytes": (
                    path.stat().st_size
                ),
                "sha256": (
                    sha256_file(
                        path
                    )
                ),
            }
        )

integrity_manifest = pd.DataFrame(
    manifest_rows
)

integrity_manifest.to_csv(
    RESULTS_DIR
    / "05_output_integrity_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Final validation: PASSED"
)

display(
    final_validation
)

print(
    "Files registered before manifest creation:",
    len(
        integrity_manifest
    ),
)

In [ ]:
# ============================================================
# 17. Export to Drive and final editorial ZIP
# ============================================================
DRIVE_NOTEBOOK11_ROOT = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK11_MANUSCRIPT_TABLES_FIGURES_OUTPUTS_V4"
)

if DRIVE_NOTEBOOK11_ROOT.exists():
    shutil.rmtree(
        DRIVE_NOTEBOOK11_ROOT
    )

shutil.copytree(
    OUTPUT_ROOT,
    DRIVE_NOTEBOOK11_ROOT,
)

FINAL_ZIP_PATH = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK11_MANUSCRIPT_TABLES_FIGURES_OUTPUTS_V4.zip"
)

if FINAL_ZIP_PATH.exists():
    FINAL_ZIP_PATH.unlink()

with zipfile.ZipFile(
    FINAL_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in sorted(
        OUTPUT_ROOT.rglob("*")
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=str(
                    path.relative_to(
                        OUTPUT_ROOT
                    )
                ),
            )

print(
    "Editorial output directory:",
    DRIVE_NOTEBOOK11_ROOT,
)

print(
    "Final editorial ZIP:",
    FINAL_ZIP_PATH,
)

print(
    "ZIP size (MB):",
    round(
        FINAL_ZIP_PATH.stat().st_size
        / 1e6,
        2,
    ),
)

# Uso editorial

Los archivos principales se encontrarán en:

```text
09_tables/11_MANUSCRIPT_ASSETS/
08_figures/11_MANUSCRIPT_ASSETS/
11_manuscript_text/
```

Antes de copiar una cifra al artículo:

1. Compruebe que aparece en `03_claim_register.csv`.
2. Revise la tabla fuente correspondiente.
3. Mantenga la distinción entre:
   - Intervalos bootstrap muestrales.
   - Variabilidad entre permutaciones.
   - Variabilidad Monte Carlo de perturbaciones.
4. No convierta análisis simulados de sensibilidad en afirmaciones de robustez experimental.